In [1]:
import sys
import time
import numpy as np
import pandas as pd
import faiss
from pathlib import Path

ROOT = Path(r"d:\ML Project - E-commerce product search with semantic ranking")
DATA = ROOT / "data"
sys.path.append(str(ROOT))

from src.evaluate import evaluate_run

corpus = pd.read_parquet(DATA / "corpus.parquet")
queries = pd.read_parquet(DATA / "queries.parquet")
qrels = pd.read_parquet(DATA / "qrels.parquet")

corpus_emb = np.load(DATA / "corpus_embeddings.npy")
query_emb = np.load(DATA / "query_embeddings.npy")

print("Corpus embeddings:", corpus_emb.shape)
print("Query embeddings:", query_emb.shape)

Corpus embeddings: (40063, 384)
Query embeddings: (500, 384)


In [2]:
dim = corpus_emb.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(corpus_emb.astype("float32"))

print("Vectors in index:", index.ntotal)

Vectors in index: 40063


In [3]:
faiss.write_index(index, str(DATA / "products.faiss"))
print("Index saved!")

Index saved!


In [4]:
product_ids = corpus["product_id"].values

def search_semantic_by_vector(qvec, top_k=10):
    qvec = np.asarray(qvec, dtype="float32").reshape(1, -1)
    scores, idx = index.search(qvec, top_k)
    return [(product_ids[i], float(s)) for i, s in zip(idx[0], scores[0])]

def show_semantic(query_index, top_k=5):
    row = queries.iloc[query_index]
    print("QUERY:", row["query"], "\n")
    labels = qrels[qrels["query_id"] == row["query_id"]].set_index("product_id")["esci_label"]
    for pid, score in search_semantic_by_vector(query_emb[query_index], top_k):
        title = corpus.loc[corpus["product_id"] == pid, "product_title"].iloc[0]
        label = labels.get(pid, "not judged")
        print(f"[{label:>10}] {score:.3f}  {title[:80]}")

show_semantic(0)

QUERY: inflatable dildo 

[         E] 0.592  Trinity Vibes Inflatable Suction Cup Dildo
[not judged] 0.569  XHYCPY 16' Inflatable Outdoor Projector Movie Screen - Package with Rope, Blower
[not judged] 0.542  Jasonwell Inflatable Dinosaur Pool Float for Boys Girls Adults 124'' Giant T-Rex
[not judged] 0.508  Inflatable Twin Baby Pool Float Infant Double Seat Floaty 2 Kids Swimming Rings
[         S] 0.504  10" Monster Silicone Realistic Dildo with Dual Density Anal Dildo Huge Suction C


In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/multi-qa-MiniLM-L6-cos-v1", device="cpu")
model.max_seq_length = 128

def search_semantic(query, top_k=10):
    qvec = model.encode(query, normalize_embeddings=True)
    return search_semantic_by_vector(qvec, top_k)

def show_text_query(query, top_k=5):
    print("QUERY:", query, "\n")
    for pid, score in search_semantic(query, top_k):
        title = corpus.loc[corpus["product_id"] == pid, "product_title"].iloc[0]
        print(f"{score:.3f}  {title[:90]}")
    print()

show_text_query("something to keep my coffee hot")
show_text_query("sofa for small room")
show_text_query("running shoes for men")

d:\ML Project - E-commerce product search with semantic ranking\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 779.95it/s]


QUERY: something to keep my coffee hot 

0.636  Wemordu Smart Coffee Mug Warmer, Coffee Cup Warmer for Desk with Auto Shut Off, Electric B
0.548  12oz Insulated Coffee Travel Mug Stainless Steel Vacuum Coffee Cup Leakproof with Screw Li
0.518  Stanley Camp Percolator w/ Silicone Cool Grip - Easy Carry, 6 Cup Stainless Steel Coffee P
0.506  Best Stainless Steel Coffee Thermos, BPA Free, New Triple Wall Insulated, Hot & Cold for H
0.481  Cresimo 68 Oz Stainless Steel Thermal Coffee Carafe / Double Walled Vacuum Flask / 12 Hour

QUERY: sofa for small room 

0.636  Best Choice Products Tufted Faux Leather 3-Seat L-Shape Sectional Sofa Couch Set w/Chaise 
0.611  Amazon Brand – Rivet Midtown Contemporary Upholstered Sofa Couch, 92.1"W, Cream
0.610  Amazon Brand – Rivet Revolve Modern Upholstered Sofa with Reversible Sectional Chaise, 80"
0.606  HONBAY Reversible Sectional Sofa Couch L-Shape Sofa Convertible Couch for Living Room 4-se
0.598  Acanva Luxury Modern Style Living Room Upholstery C

In [6]:
start = time.time()
for qtext in queries["query"].head(20):
    search_semantic(qtext, 10)
avg = (time.time() - start) / 20
print(f"Average time per query (encode + search): {avg*1000:.0f} ms")

Average time per query (encode + search): 54 ms


In [7]:
scores, idx = index.search(query_emb.astype("float32"), 100)

rows = []
for qi, qid in enumerate(queries["query_id"]):
    for rank in range(100):
        rows.append((qid, product_ids[idx[qi, rank]], rank + 1, float(scores[qi, rank])))

sem_run = pd.DataFrame(rows, columns=["query_id", "product_id", "rank", "score"])
sem_run.to_parquet(DATA / "run_semantic.parquet", index=False)

print("Saved! Rows:", len(sem_run))

Saved! Rows: 50000


In [8]:
sem_scores = evaluate_run(sem_run, qrels, k=10)
print("Semantic:", sem_scores)

bm25_run = pd.read_parquet(DATA / "run_bm25.parquet")
bm25_scores = evaluate_run(bm25_run, qrels, k=10)
print("BM25:    ", bm25_scores)

Semantic: {'Recall@10': 0.336, 'MRR@10': 0.7456, 'NDCG@10': 0.4986, 'queries_evaluated': 500}
BM25:     {'Recall@10': 0.3362, 'MRR@10': 0.7332, 'NDCG@10': 0.4952, 'queries_evaluated': 500}


In [9]:
results = pd.read_csv(DATA / "results.csv")
results = results[results["Method"] != "Semantic"]
results = pd.concat([results, pd.DataFrame([{"Method": "Semantic", **sem_scores}])], ignore_index=True)
results.to_csv(DATA / "results.csv", index=False)
results

,Method,Recall@10,MRR@10,NDCG@10,queries_evaluated
0,BM25,0.3362,0.7332,0.4952,500
1,Semantic,0.3360,0.7456,0.4986,500


In [10]:
from src.evaluate import build_qrels_dict, build_run_dict, dcg
import numpy as np

qrels_d = build_qrels_dict(qrels)
bm_d = build_run_dict(bm25_run)
sm_d = build_run_dict(sem_run)

def ndcg_q(run_d, qid, k=10):
    judged = qrels_d[qid]
    gains = [judged.get(p, 0) for p in run_d.get(qid, [])[:k]]
    ideal = dcg(sorted(judged.values(), reverse=True)[:k])
    return dcg(gains) / ideal if ideal > 0 else 0.0

diff = pd.DataFrame({
    "query_id": list(qrels_d.keys()),
    "bm25": [ndcg_q(bm_d, q) for q in qrels_d],
    "semantic": [ndcg_q(sm_d, q) for q in qrels_d],
})
diff["gain"] = diff["semantic"] - diff["bm25"]
diff = diff.merge(queries, on="query_id")

print("Semantic wins:", (diff["gain"] > 0.05).sum())
print("BM25 wins:    ", (diff["gain"] < -0.05).sum())
print("Roughly equal:", (diff["gain"].abs() <= 0.05).sum())

print("\nSemantic much better:")
display(diff.sort_values("gain", ascending=False).head(8)[["query", "bm25", "semantic"]])

print("\nBM25 much better:")
display(diff.sort_values("gain").head(8)[["query", "bm25", "semantic"]])

Semantic wins: 191
BM25 wins:     180
Roughly equal: 129

Semantic much better:


,query,bm25,semantic
183,flesh light,0.000000,0.718363
364,remove stain from bath tub,0.063621,0.661831
188,free,0.000000,0.595230
215,grip 6 buckle,0.000000,0.569788
452,tv for cars,0.000000,0.554323
62,avengers books for kids 3-5,0.331091,0.878807
225,hinatab,0.000000,0.533755
226,hook,0.078398,0.602506



BM25 much better:


,query,bm25,semantic
167,entertainment stand for tv up to 90,1.000000,0.000000
45,acudraw pro,0.732971,0.000000
362,red skull corset,0.717376,0.000000
38,95 length grey block curtains,0.682676,0.100644
196,garage storage shelves 48 x 72 x 24,0.728153,0.162107
110,carburetor jets for 350cc,0.813371,0.253112
319,npodm iv,0.781304,0.231330
56,aqua blue vinyl,0.882211,0.350268
